# Crear `competencia_01.parquet` a partir de `competencia_01.csv`

Todos los notebooks de `submits/` leen `../competencia_01.parquet`. Este notebook lo crea desde el CSV de la cátedra
(`competencia_01.csv`, en la carpeta raíz del repo) con **dos cambios**:

1. **`clase_ternaria` recalculada con la definición estricta**: BAJA+1 = el cliente no está el mes siguiente;
   BAJA+2 = está el mes siguiente pero no el subsiguiente; CONTINUA = está en los dos. 202107 solo puede ser BAJA+1
   (o vacío) y 202108 queda vacío. Respecto del CSV, 29 filas pasan de CONTINUA a BAJA (clientes que se van y vuelven).
2. **`ccajas_depositos` = NA en 202105** (en ese mes viene todo en cero: dato roto, recomendación del profesor).

Si `competencia_01.parquet` ya existe, **no lo pisa**: verifica que lo que se genera es idéntico (`DataFrame.equals`).
Tiempo: ~1 min.

In [1]:
import os
import numpy as np
import pandas as pd

ENTRADA = "../competencia_01.csv"
SALIDA  = "../competencia_01.parquet"

df = pd.read_csv(ENTRADA)
df = df.drop(columns=["clase_ternaria"])            # se recalcula abajo

# Mes como número correlativo (202103 -> 2021*12 + 3) para poder sumar 1 y 2 meses
mes_idx = ((df["foto_mes"] // 100) * 12 + df["foto_mes"] % 100).values
cliente = df["numero_de_cliente"].values
presentes = set(zip(cliente, mes_idx))               # pares (cliente, mes) que existen
esta_mes1 = np.array([(c, m + 1) in presentes for c, m in zip(cliente, mes_idx)])   # ¿está el mes siguiente?
esta_mes2 = np.array([(c, m + 2) in presentes for c, m in zip(cliente, mes_idx)])   # ¿y el subsiguiente?

ultimo = mes_idx.max()
clase = np.where(~esta_mes1, "BAJA+1", np.where(~esta_mes2, "BAJA+2", "CONTINUA")).astype(object)
clase[mes_idx == ultimo] = None                      # 202108: no hay futuro
clase[(mes_idx == ultimo - 1) & esta_mes1] = None    # 202107: solo se sabe si es BAJA+1
df = df.assign(clase_ternaria=clase)

df.loc[df["foto_mes"] == 202105, "ccajas_depositos"] = np.nan   # dato roto en 202105

print(df.groupby("foto_mes")["clase_ternaria"].value_counts(dropna=False).unstack())

clase_ternaria  BAJA+1  BAJA+2  CONTINUA       NaN
foto_mes                                          
202103          1019.0   960.0  160921.0       NaN
202104           964.0  1139.0  161181.0       NaN
202105          1143.0   870.0  161755.0       NaN
202106           874.0  1098.0  162142.0       NaN
202107          1103.0     NaN       NaN  163245.0
202108             NaN     NaN       NaN  164647.0


C:\Users\keila\AppData\Local\Temp\ipykernel_13868\3050616973.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.assign(clase_ternaria=clase)


In [2]:
if os.path.exists(SALIDA):
    print("Ya existe: ¿es idéntico al que se genera?", df.equals(pd.read_parquet(SALIDA)))
else:
    df.to_parquet(SALIDA, index=False)
    print("Creado:", SALIDA, df.shape)

Ya existe: ¿es idéntico al que se genera? True
